# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [13]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY') 

### Example 1

#### Create our QandA application

In [15]:
import sys
print(sys.executable)

c:\Users\Shrabani P\IronHack\Week8_Day2\Lab_Week8_Day2\.venv\Scripts\python.exe


In [16]:
%pip install langchain langchain-openai langchain-huggingface

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
%pip install langchain-classic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
%pip install langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
from langchain_classic.chains import RetrievalQA, LLMChain
from langchain_openai import OpenAI, ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_classic.indexes import VectorstoreIndexCreator
from langchain_community.vectorstores import DocArrayInMemorySearch

In [20]:
file = r'C:\Users\Shrabani P\IronHack\Week8_Day2\Lab_Week8_Day2\lab-langchain-evaluation\data\OutdoorClothingCatalog_1000.csv'

loader = CSVLoader(
    file_path=file,
    encoding="utf-8"
)

data = loader.load()

print(len(data))
print(data[0])

1000
page_content=': 0
name: Women's Campside Oxfords
description: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. 

Size & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. 

Specs: Approx. weight: 1 lb.1 oz. per pair. 

Construction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. 

Questions? Please contact us for any inquiries.' metadata={'source': 'C:\\Users\\Shrabani P\\IronHack\\Week8_Day2\\Lab_Week8_Day2\\lab-langchain-evaluation\\data\\OutdoorClothingCatalog_1000.csv', 'row': 0}


In [21]:
 !pip install --upgrade --force-reinstall sentence-transformers

^C


In [27]:
%pip install docarray

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import docarray

print(docarray.__version__)

0.40.1


In [29]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cpu'})
).from_loaders([loader])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5112.58it/s]


In [30]:
llm = ChatOpenAI(temperature = 0.0)
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=index.vectorstore.as_retriever(), 
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

#### Coming up with test datapoints

In [31]:
data[10]

Document(metadata={'source': 'C:\\Users\\Shrabani P\\IronHack\\Week8_Day2\\Lab_Week8_Day2\\lab-langchain-evaluation\\data\\OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\r\n\r\nSize & Fit\r\n- Pants are Favorite Fit: Sits lower on the waist.\r\n- Relaxed Fit: Our most generous fit sits farthest from the body.\r\n\r\nFabric & Care\r\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\r\n\r\nAdditional Features\r\n- Relaxed fit top with raglan sleeves and rounded hem.\r\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\r\n\r\nImported.")

In [32]:
data[11]

Document(metadata={'source': 'C:\\Users\\Shrabani P\\IronHack\\Week8_Day2\\Lab_Week8_Day2\\lab-langchain-evaluation\\data\\OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

#### Hard-coded examples

In [33]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field

examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

# Define the prompt template
prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do the Cozy Comfort Pullover Set have side pockets?\n"
             "   Answer: Yes\n"
             "2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?\n"
             "   Answer: The DownTek collection\n"
             "Query: {query}\n"
             "Answer:"
)

# Define the output model
class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

# Create the output parser
class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        # Split the response to get the answer
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)

# Initialize the LLM
# llm = OpenAI()
llm = ChatOpenAI()

# Create the LLMChain
llm_chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    output_parser=AnswerOutputParser()
)

# Example query
query = "Is the Cozy Comfort Pullover Set available in different colors?"

# Run the chain
result = llm_chain.run({"query": query})

# Print the result
print(result)


C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\1180835104.py:46: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(
C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\1180835104.py:56: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = llm_chain.run({"query": query})


answer='No, the Cozy Comfort Pullover Set is only available in one color: grey.'


#### LLM-Generated examples

In [34]:
from langchain_classic.evaluation.qa import QAGenerateChain

In [35]:
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI())

In [36]:
llm_chain = LLMChain(llm=llm, prompt=prompt_template)

In [37]:
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\3125874183.py:1: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  new_examples = example_gen_chain.apply_and_parse(
c:\Users\Shrabani P\IronHack\Week8_Day2\Lab_Week8_Day2\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:550: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")


In [38]:
new_examples[0]

{'qa_pairs': {'query': "What is the approximate weight of the Women's Campside Oxfords per pair?",
  'answer': "The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz."}}

In [39]:
data[0]

Document(metadata={'source': 'C:\\Users\\Shrabani P\\IronHack\\Week8_Day2\\Lab_Week8_Day2\\lab-langchain-evaluation\\data\\OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \r\n\r\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \r\n\r\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \r\n\r\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \r\n\r\nQuestions? Please contact us for any inquiries.")

In [40]:
d_flattened = [data['qa_pairs'] for data in new_examples]
d_flattened

[{'query': "What is the approximate weight of the Women's Campside Oxfords per pair?",
  'answer': "The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz."},
 {'query': 'What are some key features of the Recycled Waterhog Dog Mat, Chevron Weave mentioned in the document?',
  'answer': 'Some key features of the Recycled Waterhog Dog Mat include being made from recycled plastic materials, having thick and thin fibers for scraping dirt and absorbing water, drying quickly, resisting fading and shedding, and being suitable for indoor and outdoor use.'},
 {'query': "What are some key features of the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece as described in the document?",
  'answer': "Some key features of the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece include bright colors, ruffles, exclusive whimsical prints, four-way-stretch and chlorine-resistant fabric, UPF 50+ rated fabric for sun protection, crossover no-slip straps, fully line

#### Combine examples

In [41]:
# examples += new_example
examples += d_flattened

In [42]:
examples[0]

{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'answer': 'Yes'}

In [43]:
qa.invoke(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'result': 'Yes, the Cozy Comfort Pullover Set has side seam pockets and a back zip pocket, as well as two elastic mesh water bottle pockets.'}

### Manual Evaluation - Fun part

In [44]:
import langchain
langchain.debug = True

In [45]:
qa.invoke(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'result': 'Based on the provided context, the Cozy Comfort Pullover Set does not have side pockets. It has side seam pockets, a back zip pocket, two elastic mesh water bottle pockets, and a top compartment with a pocket with a double-seal zipper for quick access.'}

In [46]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [47]:
examples += d_flattened

In [48]:
examples

[{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
  'answer': 'Yes'},
 {'query': 'What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection'},
 {'query': "What is the approximate weight of the Women's Campside Oxfords per pair?",
  'answer': "The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz."},
 {'query': 'What are some key features of the Recycled Waterhog Dog Mat, Chevron Weave mentioned in the document?',
  'answer': 'Some key features of the Recycled Waterhog Dog Mat include being made from recycled plastic materials, having thick and thin fibers for scraping dirt and absorbing water, drying quickly, resisting fading and shedding, and being suitable for indoor and outdoor use.'},
 {'query': "What are some key features of the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece as described in the document?",
  'answer': "Some key features of the Infant and Toddler 

In [49]:
predictions = qa.batch(examples)



> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.


In [50]:
predictions

[{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
  'answer': 'Yes',
  'result': 'Based on the provided context, the Cozy Comfort Pullover Set does not have side pockets. It has side seam pockets, a back zip pocket, two elastic mesh water bottle pockets, and a top compartment with a pocket with a double-seal zipper for quick access.'},
 {'query': 'What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection',
  'result': 'The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.'},
 {'query': "What is the approximate weight of the Women's Campside Oxfords per pair?",
  'answer': "The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz.",
  'result': "The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz."},
 {'query': 'What are some key features of the Recycled Waterhog Dog Mat, Chevron Weave mentioned in the document?',
  'answer': 'Some ke

In [51]:
from langchain_classic.evaluation.qa import QAEvalChain

In [52]:
llm = ChatOpenAI(temperature=0)
eval_chain = QAEvalChain.from_llm(llm)

In [53]:
graded_outputs = eval_chain.evaluate(examples, predictions)

c:\Users\Shrabani P\IronHack\Week8_Day2\Lab_Week8_Day2\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:550: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")


In [54]:
graded_outputs

[{'results': 'INCORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'}]

In [55]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    # print("Predicted Grade: " + graded_outputs[i]['text'])
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set        have side pockets?
Real Answer: Yes
Predicted Answer: Based on the provided context, the Cozy Comfort Pullover Set does not have side pockets. It has side seam pockets, a back zip pocket, two elastic mesh water bottle pockets, and a top compartment with a pocket with a double-seal zipper for quick access.

Example 1:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.

Example 2:
Question: What is the approximate weight of the Women's Campside Oxfords per pair?
Real Answer: The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz.
Predicted Answer: The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz.

Example 3:
Question: What are some key features of the Recycled Waterhog Dog Mat, Chevron Weave mentioned in the

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [56]:
import os

print(os.getcwd())

c:\Users\Shrabani P\IronHack\Week8_Day2\Lab_Week8_Day2\lab-langchain-evaluation


In [57]:
from langchain_huggingface import HuggingFaceEmbeddings
loader = TextLoader(
    "data/nyc_text.txt",
    encoding="utf-8"
)
index = VectorstoreIndexCreator(embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cpu'})).from_loaders([loader])


llm = ChatOpenAI(temperature= 0)
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=index.vectorstore.as_retriever(),
    return_source_documents=True,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6446.61it/s]
C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\3746465160.py:6: UserWarning: Using InMemoryVectorStore as the default vectorstore.This memory store won't persist data. You should explicitlyspecify a VectorStore when using VectorstoreIndexCreator
  index = VectorstoreIndexCreator(embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cpu'})).from_loaders([loader])


In [58]:
# testing it out

question = "How did New York City get its name?"
result = qa_chain.invoke({"query": question})
result["result"]

'New York City was originally named New Amsterdam by Dutch colonists in 1626. When the city came under British control in 1664, it was renamed New York after King Charles II of England granted the lands to his brother, the Duke of York. The city has been continuously named New York since November 1674.'

In [59]:
result

{'query': 'How did New York City get its name?',
 'result': 'New York City was originally named New Amsterdam by Dutch colonists in 1626. When the city came under British control in 1664, it was renamed New York after King Charles II of England granted the lands to his brother, the Duke of York. The city has been continuously named New York since November 1674.',
 'source_documents': [Document(id='3cc3e14c-8139-4df3-b625-a63914b9512e', metadata={'source': 'data/nyc_text.txt'}, page_content='The city and its metropolitan area constitute the premier gateway for legal immigration to the United States. As many as 800 languages are spoken in New York, making it the most linguistically diverse city in the world. New York City is home to more than 3.2 million residents born outside the U.S., the largest foreign-born population of any city in the world as of 2016.New York City traces its origins to a trading post founded on the southern tip of Manhattan Island by Dutch colonists in approximate

Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [60]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [61]:
examples

[{'query': 'What is the population of New York City as of 2020?',
  'ground_truths': ['8,804,190']},
 {'query': 'Which borough of New York City has the highest population?',
  'ground_truths': ['Brooklyn']},
 {'query': 'What is the economic significance of New York City?',
  'ground_truths': ["New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter."]},
 {'query': 'How did New York City

#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain). 
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

In [62]:
result = qa_chain.invoke({"query": eval_questions[1]})
result["result"]

'Manhattan (New York County) has the highest population density of any borough in New York City.'

In [63]:
key_mapping = {
    "query": "question",
    "result": "answer",
    "source_documents": "contexts"
}

result_updated = {}
for old_key, new_key in key_mapping.items():
    if old_key in result:
        result_updated[new_key] = result[old_key]


In [64]:
result_updated

{'question': 'Which borough of New York City has the highest population?',
 'answer': 'Manhattan (New York County) has the highest population density of any borough in New York City.',
 'contexts': [Document(id='4c6fe62e-c9cb-4dd0-8a71-52407f77ad4f', metadata={'source': 'data/nyc_text.txt'}, page_content="New York City is the most populous city in the United States, with 8,804,190 residents incorporating more immigration into the city than outmigration since the 2010 United States census. More than twice as many people live in New York City as compared to Los Angeles, the second-most populous U.S. city; and New York has more than three times the population of Chicago, the third-most populous U.S. city. New York City gained more residents between 2010 and 2020 (629,000) than any other U.S. city, and a greater amount than the total sum of the gains over the same decade of the next four largest U.S. cities, Los Angeles, Chicago, Houston, and Phoenix, Arizona combined. New York City's popu

In [65]:
!pip install --no-cache-dir recordclass

     ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
      --------------------------------------- 0.0/1.3 MB 640.0 kB/s eta 0:00:03
     ------ --------------------------------- 0.2/1.3 MB 2.8 MB/s eta 0:00:01
     -------------------------------- ------- 1.1/1.3 MB 8.5 MB/s eta 0:00:01
     ---------------------------------------- 1.3/1.3 MB 10.6 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build recordclass


  error: subprocess-exited-with-error
  
  × Building wheel for recordclass (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [97 lines of output]
      C:\Users\Shrabani P\AppData\Local\Temp\pip-build-env-6psb0bp5\overlay\Lib\site-packages\setuptools\config\_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
      !!
      
              ********************************************************************************
              Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).
      
              By 2027-Feb-18, you need to update your project and remove deprecated calls
              or your builds will no longer be supported.
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ****************************************

In [66]:
 !pip install ragas==0.1.9

  Using cached ragas-0.1.9-py3-none-any.whl.metadata (5.2 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
Using cached ragas-0.1.9-py3-none-any.whl (86 kB)
Using cached fsspec-2026.6.0-py3-none-any.whl (203 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.7.0
    Uninstalling fsspec-2026.7.0:
      Successfully uninstalled fsspec-2026.7.0
  Attempting uninstall: ragas
    Found existing installation: ragas 0.4.3
    Uninstalling ragas-0.4.3:
      Successfully uninstalled ragas-0.4.3



[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [67]:
import importlib.metadata as md

packages = [
    "ragas",
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-classic"
]

for package in packages:
    try:
        print(package, ":", md.version(package))
    except md.PackageNotFoundError:
        print(package, ": NOT INSTALLED")

ragas : 0.1.9
langchain : 1.3.14
langchain-core : 1.5.3
langchain-community : 0.4.1
langchain-classic : 1.0.8


In [68]:
%pip show ragas

Name: ragas
Version: 0.1.9
Summary: 
Home-page: 
Author: 
Author-email: 
License: 
Location: c:\Users\Shrabani P\IronHack\Week8_Day2\Lab_Week8_Day2\.venv\Lib\site-packages
Requires: appdirs, datasets, langchain, langchain-community, langchain-core, langchain-openai, nest-asyncio, numpy, openai, pysbd, tiktoken
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [69]:
%pip check

No broken requirements found.
Note: you may need to restart the kernel to use updated packages.


In [70]:
%pip install -U ragas

  Using cached ragas-0.4.3-py3-none-any.whl.metadata (23 kB)
Using cached ragas-0.4.3-py3-none-any.whl (466 kB)
  Attempting uninstall: ragas
    Found existing installation: ragas 0.1.9
    Uninstalling ragas-0.1.9:
      Successfully uninstalled ragas-0.1.9
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [71]:
import importlib.metadata as md

print(md.version("ragas"))

0.4.3


In [72]:
%pip install "langchain-community==0.4.1"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [73]:
from ragas import evaluate

print("Ragas works")

Ragas works


In [74]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
)

metrics = [
    faithfulness,
    answer_relevancy,
    context_recall,
]

C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\2930750059.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\2930750059.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\2930750059.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (


In [75]:
from ragas import evaluate

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
)

metrics = [
    faithfulness,
    answer_relevancy,
    context_recall,
]

print("Ragas metrics loaded successfully")

Ragas metrics loaded successfully


C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\2950659820.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\2950659820.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\2950659820.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (


1. `__call__()`

Directly run the evaluation chain with the results from the QA chain. Do note that metrics like context_relevancy and faithfulness require the `source_documents` to be present.

In [76]:
# Recheck the result that we are going to validate.

result = qa_chain.invoke({"query": eval_questions[1]})
result["result"]

'Manhattan (New York County) has the highest population density of any borough in New York City.'

In [78]:
result.keys()

dict_keys(['query', 'result', 'source_documents'])

**Faithfulness**

In [79]:
from ragas import EvaluationDataset
from ragas.dataset_schema import SingleTurnSample

sample = SingleTurnSample(
    user_input=result["query"],
    response=result["result"],
    retrieved_contexts=[
        doc.page_content for doc in result["source_documents"]
    ]
)

dataset = EvaluationDataset(samples=[sample])

In [80]:
from ragas import evaluate
from ragas.metrics import faithfulness

eval_result = evaluate(
    dataset=dataset,
    metrics=[faithfulness]
)

C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\1892571174.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness
Evaluating: 100%|██████████| 1/1 [00:05<00:00,  5.69s/it]


In [83]:
eval_result = evaluate(
    dataset=dataset,
    metrics=[faithfulness]
)

eval_result.to_pandas()

Evaluating: 100%|██████████| 1/1 [00:08<00:00,  8.61s/it]


,user_input,retrieved_contexts,response,faithfulness
0,Which borough of New York City has the highest...,[New York City is the most populous city in th...,Manhattan (New York County) has the highest po...,1.0


High faithfulness_score means that there are exact consistency between the source documents and the answer.

You can check lower faithfulness scores by changing the result (answer from LLM) or source_documents to something else.

In [84]:
from ragas import EvaluationDataset
from ragas.dataset_schema import SingleTurnSample

fake_result = result.copy()
fake_result["result"] = "we are the champions"

fake_sample = SingleTurnSample(
    user_input=fake_result["query"],
    response=fake_result["result"],
    retrieved_contexts=[
        doc.page_content for doc in fake_result["source_documents"]
    ]
)

fake_dataset = EvaluationDataset(samples=[fake_sample])

In [85]:
from ragas import evaluate
from ragas.metrics import faithfulness

eval_result = evaluate(
    dataset=fake_dataset,
    metrics=[faithfulness]
)

eval_result.to_pandas()

C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\2539957413.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness
Evaluating: 100%|██████████| 1/1 [00:02<00:00,  2.90s/it]


,user_input,retrieved_contexts,response,faithfulness
0,Which borough of New York City has the highest...,[New York City is the most populous city in th...,we are the champions,0.0


In [86]:
fake_result = result.copy()
fake_result["result"] = "we are the champions"

fake_sample = SingleTurnSample(
    user_input=fake_result["query"],
    response=fake_result["result"],
    retrieved_contexts=[
        doc.page_content for doc in fake_result["source_documents"]
    ]
)

fake_dataset = EvaluationDataset(samples=[fake_sample])

eval_result = evaluate(
    dataset=fake_dataset,
    metrics=[faithfulness]
)

eval_result.to_pandas()

Evaluating: 100%|██████████| 1/1 [00:04<00:00,  4.95s/it]


,user_input,retrieved_contexts,response,faithfulness
0,Which borough of New York City has the highest...,[New York City is the most populous city in th...,we are the champions,0.0


**Context Relevancy**

In [88]:
from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import context_recall

sample = SingleTurnSample(
    user_input=result["query"],
    response=result["result"],
    retrieved_contexts=[
        doc.page_content for doc in result["source_documents"]
    ],
    reference="PUT THE EXPECTED / GROUND-TRUTH ANSWER HERE"
)

dataset = EvaluationDataset(samples=[sample])

eval_result = evaluate(
    dataset=dataset,
    metrics=[context_recall]
)

eval_result.to_pandas()

C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\3771372199.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_recall
Evaluating: 100%|██████████| 1/1 [00:01<00:00,  1.58s/it]


,user_input,retrieved_contexts,response,reference,context_recall
0,Which borough of New York City has the highest...,[New York City is the most populous city in th...,Manhattan (New York County) has the highest po...,PUT THE EXPECTED / GROUND-TRUTH ANSWER HERE,0.0


In [89]:
eval_result = evaluate(
    dataset=dataset,
    metrics=[context_recall]
)

eval_result.to_pandas()

Evaluating: 100%|██████████| 1/1 [00:01<00:00,  1.33s/it]


,user_input,retrieved_contexts,response,reference,context_recall
0,Which borough of New York City has the highest...,[New York City is the most populous city in th...,Manhattan (New York County) has the highest po...,PUT THE EXPECTED / GROUND-TRUTH ANSWER HERE,0.0


In [90]:
print(result.keys())
print(eval_questions[1])

dict_keys(['query', 'result', 'source_documents'])
Which borough of New York City has the highest population?


In [91]:
print(eval_answers[1])

Brooklyn


High context_recall_score means that the ground truth is present in the source documents.

You can check lower context recall scores by changing the source_documents to something else.

In [96]:
from langchain_core.documents import Document
from ragas import EvaluationDataset, evaluate
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import context_recall

# Copy the original QA result
fake_result = result.copy()

# Replace retrieved context with irrelevant/fake context
fake_result["source_documents"] = [
    Document(page_content="I love christmas")
]

# Create a Ragas sample
fake_sample = SingleTurnSample(
    user_input=fake_result["query"],
    response=fake_result["result"],
    retrieved_contexts=[
        doc.page_content for doc in fake_result["source_documents"]
    ],
    reference="YOUR CORRECT EXPECTED ANSWER"
)

# Create evaluation dataset
fake_dataset = EvaluationDataset(samples=[fake_sample])

# Evaluate context recall
eval_result = evaluate(
    dataset=fake_dataset,
    metrics=[context_recall]
)

# Display score
eval_result.to_pandas()

C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\2580116542.py:4: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_recall
Evaluating: 100%|██████████| 1/1 [00:01<00:00,  1.69s/it]


,user_input,retrieved_contexts,response,reference,context_recall
0,Which borough of New York City has the highest...,[I love christmas],Manhattan (New York County) has the highest po...,YOUR CORRECT EXPECTED ANSWER,0.0


2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [99]:
# Run the queries as a batch
predictions = qa_chain.batch(examples)

In [101]:
print(predictions[0].keys())
print(predictions[0])

dict_keys(['query', 'ground_truths', 'result', 'source_documents'])
{'query': 'What is the population of New York City as of 2020?', 'ground_truths': ['8,804,190'], 'result': 'The population of New York City as of 2020 is 8,804,190 residents.', 'source_documents': [Document(id='4c6fe62e-c9cb-4dd0-8a71-52407f77ad4f', metadata={'source': 'data/nyc_text.txt'}, page_content="New York City is the most populous city in the United States, with 8,804,190 residents incorporating more immigration into the city than outmigration since the 2010 United States census. More than twice as many people live in New York City as compared to Los Angeles, the second-most populous U.S. city; and New York has more than three times the population of Chicago, the third-most populous U.S. city. New York City gained more residents between 2010 and 2020 (629,000) than any other U.S. city, and a greater amount than the total sum of the gains over the same decade of the next four largest U.S. cities, Los Angeles, Ch

In [103]:
predictions = qa_chain.batch(examples)

print("EXAMPLE:")
print(examples[0])

print("\nPREDICTION:")
print(predictions[0])

print("\nPREDICTION KEYS:")
print(predictions[0].keys())

EXAMPLE:
{'query': 'What is the population of New York City as of 2020?', 'ground_truths': ['8,804,190']}

PREDICTION:
{'query': 'What is the population of New York City as of 2020?', 'ground_truths': ['8,804,190'], 'result': 'The population of New York City as of 2020 was 8,804,190 residents.', 'source_documents': [Document(id='4c6fe62e-c9cb-4dd0-8a71-52407f77ad4f', metadata={'source': 'data/nyc_text.txt'}, page_content="New York City is the most populous city in the United States, with 8,804,190 residents incorporating more immigration into the city than outmigration since the 2010 United States census. More than twice as many people live in New York City as compared to Los Angeles, the second-most populous U.S. city; and New York has more than three times the population of Chicago, the third-most populous U.S. city. New York City gained more residents between 2010 and 2020 (629,000) than any other U.S. city, and a greater amount than the total sum of the gains over the same decade o

In [107]:
from ragas import EvaluationDataset
from ragas.dataset_schema import SingleTurnSample

samples = []

for prediction in predictions:
    sample = SingleTurnSample(
        user_input=prediction["query"],
        response=prediction["result"],
        retrieved_contexts=[
            doc.page_content
            for doc in prediction["source_documents"]
        ],
        reference=prediction["ground_truths"][0]
    )

    samples.append(sample)

evaluation_dataset = EvaluationDataset(samples=samples)

In [108]:
from ragas import evaluate
from ragas.metrics import context_recall

print("evaluating...")

r = evaluate(
    dataset=evaluation_dataset,
    metrics=[context_recall]
)

r

C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_23180\3175280211.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_recall


evaluating...


Evaluating: 100%|██████████| 5/5 [00:13<00:00,  2.75s/it]


{'context_recall': 0.7333}